# Testing Quadratic Assignment Problem (QAP)

In [ ]:
using Revise, Quicopt, QuicoptUtils, LinearAlgebra
using NPZ, Printf, PythonCall
import Random

In [ ]:
set_print_decimals(2, scientific=true)
use_std_mpl_style()

In [ ]:
function qap_to_ising(f::AbstractMatrix, d::AbstractMatrix;
                      B::Real = 1.0,
                      A::Real = B * size(f,1)^2 * maximum(abs, f) * maximum(abs, d) + 1)
    N = size(f, 1)
    @assert size(f) == (N, N) == size(d)

    n   = N^2
    idx(i, k) = (i-1)*N + k

    F = vec(sum(f; dims=2))
    D = vec(sum(d; dims=2))
    h = vec(fill(float(A) * (2 - N), N, N) .- (B/2) .* F .* D')

    J = zeros(n, n)
    for i in 1:N, k in 1:N
        ik = idx(i, k)
        for j in 1:N, l in 1:N
            jl = idx(j, l)
            jl <= ik && continue
            val  = (B/2) * f[i,j] * d[k,l]
            val += (A/2) * (i == j && k != l)
            val += (A/2) * (k == l && i != j)
            J[ik, jl] = J[jl, ik] = val
        end
    end

    return h, J
end

using Combinatorics: permutations

function qap_cost(f, d, π)
    N = length(π)
    sum(f[i,j] * d[π[i], π[j]] for i in 1:N, j in 1:N)
end

function ising_energy(h, J, s)
    # sum_{i<j} J[i,j]*s[i]*s[j], not dot(s, J*s) which double-counts
    dot(h, s) + dot(s, J * s) / 2
end

function validate_qap_ising(f, d; kw...)
    N = size(f, 1)
    h, J = qap_to_ising(f, d; kw..., B=1.0)
    idx(i, k) = (i-1)*N + k
    energy(s) = dot(h, s) + dot(s, J * s) / 2

    perm_results = map(permutations(1:N)) do π
        s = fill(1, N^2); s[idx.(1:N, π)] .= -1
        sum(f[i,j] * d[π[i], π[j]] for i in 1:N, j in 1:N), s, energy(s)
    end

    qap_vals   = first.(perm_results)
    ising_vals = last.(perm_results)
    println(perm_results)

    @assert norm((qap_vals .- qap_vals[1]) .- (ising_vals .- ising_vals[1])) < 1e-8 "Encoding mismatch"

    min_feasible = minimum(ising_vals)
    min_all = minimum(energy(2 .* digits(s, base=2, pad=N^2) .- 1) for s in 0:2^(N^2)-1)
    @assert min_all ≈ min_feasible "Infeasible state undercuts feasible minimum by $(min_feasible - min_all)"
    println("✓ All $(2^(N^2)) states checked. Optimal π at index $(argmin(qap_vals))")
end

In [ ]:
PATH = "../../instances/quadratic_assignment/"
filename = "C_random.npz"
f = npzread(PATH * filename)
η = 1.0  # or whatever exponent
N = size(f, 1)
d = [abs(k - l)^η for k in 1:N, l in 1:N]

h, J = qap_to_ising(f, d);

In [ ]:
# Test
f_test = [0.0 1 2; 1 0 1; 2 1 0]
d_test = float.([abs(k-l) for k in 1:3, l in 1:3])
h_test, J_test = qap_to_ising(f_test, d_test)
validate_qap_ising(f_test, d_test)

In [ ]:
@p h_test

In [ ]:
J_test

In [ ]:
T_final = 2.0^10
s_x = t -> 1.0 - t/T_final
s_z = t -> t/T_final;

In [ ]:
# solution = evolve(-h, -J, T_final, s_z);
solution = evolve(-h_test, -J_test, T_final, s_z);

In [ ]:
spin_sol = sign.(solution[end][3, :])
# bit_sol = spin_to_binary.(spin_sol)
@p spin_sol .|> Int
# @p ising_energy(h, J, spin_sol)
@p ising_energy(h_test, J_test, spin_sol)

## Testing

In [ ]:
Random.seed!(1)
cat_strength = 1.0
catalyst_xtensor =  Dict{Tuple{Int64, Vararg{Int64}}, Float64}((1,) => 0.0)
catalyst_ztensor = Dict{Tuple{Int64, Vararg{Int64}}, Float64}([(i,) => -cat_strength * randn() for i in 1:size(local_fields, 1)])

# minus the couplings because we are minimizing
cat_problem = TensorProblem(-h, -J, catalyst_xtensor, catalyst_ztensor)

In [ ]:
solution = evolve(cat_problem, T_final, s_x, s_z, t -> s_x(t) * s_z(t), t -> s_x(t) * s_z(t));

In [ ]:
spin_sol = sign.(solution[end][3, :])
bit_sol = spin_to_binary.(spin_sol)